# 電子親和力 (EA) と 電子再配列エネルギー (λ_e) の計算
指定したSMILESから、以下の4つの状態のエネルギーを計算し、垂直EA (vEA)、断熱EA (aEA)、および再配列エネルギーを算出します。

* **E(N@N)**: 中性状態での最適化構造における中性状態のエネルギー
* **E(A@N)**: 中性状態での最適化構造におけるアニオン(ラジカル)状態のエネルギー
* **E(A@A)**: アニオン状態での最適化構造におけるアニオン(ラジカル)状態のエネルギー
* **E(N@A)**: アニオン状態での最適化構造における中性状態のエネルギー

**計算式**:
* 垂直EA (vEA) = E(N@N) - E(A@N)
* 断熱EA (aEA) = E(N@N) - E(A@A)
* 緩和エネルギー1 (λ1) = E(A@N) - E(A@A)
* 緩和エネルギー2 (λ2) = E(N@A) - E(N@N)
* 電子再配列エネルギー (λ_e) = λ1 + λ2

分子量の大きい分子で実行するとメモリ不足のエラーが出る可能性有。その場合は、
- 構造最適化は、`6-31G*`
- 4つのエネルギー（中性・イオンの単点だけ）`6-31+G*`     

で行うと解決することあり、`.kernel()`の前で構造最適化済みの座標を使い、基底関数だけ`6-31+G`の分子を作り直し、構造最適化の`gto.M`と`optimize`は`6-31G*`にする


In [ ]:
import os
import cupy
import gc
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto
from gpu4pyscf.dft import rks, uks
from pyscf.geomopt.geometric_solver import optimize

# --- 設定 ---
# EAを計算したい分子のSMILESをここに貼る
SMILES = 'c1ccc(cc1)n1c(nc2ccccc12)c1cc(cc(c1)c1nc2ccccc2n1c1ccccc1)c1nc2ccccc2n1c1ccccc1' # TPBi
MOL_NAME = 'TPBi_EA'
XC_FUNCTIONAL = 'b3lyp'
BASIS_SET = '6-31+g*' # 6-31+G(d) (アニオンには+が必須)

out_dir = f"output/{MOL_NAME}"
os.makedirs(out_dir, exist_ok=True)

def release_gpu():
    """計算間にGPUメモリプールを返す。"""
    gc.collect()
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()

In [2]:
# 1. RDKitによる初期構造生成
print(f"[{MOL_NAME}] RDKitで3D構造を生成中...")
mol_rdkit = Chem.MolFromSmiles(SMILES)
mol_rdkit = Chem.AddHs(mol_rdkit)
AllChem.EmbedMolecule(mol_rdkit, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol_rdkit)

rdkit_xyz = os.path.join(out_dir, f"{MOL_NAME}_rdkit.xyz")
Chem.MolToXYZFile(mol_rdkit, rdkit_xyz)
print("RDKit構造を保存しました。")

[TPBi_EA] RDKitで3D構造を生成中...
RDKit構造を保存しました。


In [3]:
# 2. 中性状態の構造最適化 (N@N)
print("\n--- [1/4] 中性状態の構造最適化 (N@N) ---")
mol_N = gto.M(atom=rdkit_xyz, basis=BASIS_SET, charge=0, spin=0)
mf_N = rks.RKS(mol_N, xc=XC_FUNCTIONAL)
mol_N_opt = optimize(mf_N, maxsteps=100)
del mf_N
release_gpu()

# 中性状態のエネルギー計算（最適化構造でのエネルギーを取得するため、再度SP計算を実行。）
# 構造最適化と同じ直接SCFを使う（遅いがメモリは最適化時と同じ）。
mf_N_opt = rks.RKS(mol_N_opt, xc=XC_FUNCTIONAL)
E_N_N = mf_N_opt.kernel()
print(f"E(N@N) = {E_N_N:.6f} Hartree")

# 中性最適化構造の保存
xyz_N_opt = os.path.join(out_dir, f"{MOL_NAME}_Neutral_opt.xyz")
with open(xyz_N_opt, "w") as f:
    f.write(mol_N_opt.tostring(format='xyz'))

del mf_N_opt
release_gpu()

geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3809c46b095f182629549aa658ba98d0643d4db64.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    


              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********      .))))))  
          ##      .%%%%%%/      (%%%%%%,                  ,******      /)))))  
        %%%%%%      .%%%%%%#      *%%%%%%,    ,/////.       ******      )))))) 
      #%      %%      .%%%%%%/      *%%%%%%,  ////////,      *****/     ,))))) 
    #%%  %%%  %%%#      .%%%%%%/      (%%%%%%,  ///////.     /*****      ))))).
  #%%%%.      %%%%%#      /%%%%%%*      #%%%%%%   /////)     ******      ))))),
    #%%%%##%  %%%#      .%%%%%%/      (%%%%%%,  ///////.     /*****      ))))).
      ##     %%%      .%%%%%%/      *%%%%%%,  ////////.      *****/     ,))))) 
        #%%%%#      /%%%%%%/      (%%%%%%      /)/)//       ******      )))))) 
          ##      .%%%%%%/      (%%%%%%,                  *******      ))))))  
                .%%%%%%/      *%%%%%%,  **.             /*******      .))))))  
              *%%%%%%/      (%%%%%%   **


--- [1/4] 中性状態の構造最適化 (N@N) ---


243 internal coordinates being used (instead of 243 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-6
Distance 1-52
Distance 2-3
Distance 2-53
Distance 3-4
Distance 3-54
Distance 4-5
Distance 4-7
Distance 5-6
Distance 5-55
Distance 6-56
Distance 7-8
Distance 7-15
Distance 8-9
Distance 8-16
Distance 9-10
Distance 10-11
Distance 10-15
Distance 11-12
Distance 11-57
Distance 12-13
Distance 12-58
Distance 13-14
Distance 13-59
Distance 14-15
Distance 14-60
Distance 16-17
Distance 16-21
Distance 17-18
Distance 17-61
Distance 18-19
Distance 18-37
Distance 19-20
Distance 19-62
Distance 20-21
Distance 20-22
Distance 21-63
Distance 22-23
Distance 22-30
Distance 23-24
Distance 24-25
Distance 24-29
Distance 25-26
Distance 25-64
Distance 26-27
Distance 26-65
Distance 27-28
Distance 27-66
Distance 28-29
Distance 28-67
Distance 29-30
Distance 30-31
Distance 31-32
Distance 31-36
Distance 32-33
Distance 32-68
Distance 33-34
Distance 33-69
Distance 34-35
Distance 3


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   2.109723  -2.785804  -3.485128    0.000000  0.000000  0.000000
   C   3.050111  -3.494151  -2.737964    0.000000  0.000000  0.000000
   C   3.564590  -2.950334  -1.559123   -0.000000  0.000000  0.000000
   C   3.128466  -1.694082  -1.121353   -0.000000  0.000000  0.000000
   C   2.183072  -0.984526  -1.875468    0.000000  0.000000  0.000000
   C   1.678501  -1.531426  -3.056554    0.000000  0.000000  0.000000
   N   3.637099  -1.140632   0.089143    0.000000  0.000000 -0.000000
   C   2.916596  -0.448896   1.068794    0.000000  0.000000  0.000000
   N   3.723562   0.036002   1.994691    0.000000  0.000000  0.000000
   C   4.973648  -0.412737   1.680424    0.000000  0.000000  0.000000
   C   6.178356  -0.203212   2.369952    0.000000  0.000000  0.000000
   C   7.361738  -0.731635   1.846488    0.000000  0.000000  0.000000
   C   7.348841  -1.441555   0.6

...


Geometry optimization cycle 22
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   2.381804  -4.076390  -2.878013    0.003331  0.001242  0.001992
   C   2.931906  -4.534146  -1.676949    0.002950  0.001131  0.002122
   C   3.346343  -3.623479  -0.701850    0.002694  0.001054  0.002303
   C   3.212063  -2.250387  -0.934364    0.002823  0.001065  0.002317
   C   2.675006  -1.786400  -2.140374    0.003191  0.001152  0.002185
   C   2.253402  -2.702612  -3.106579    0.003431  0.001249  0.001988
   N   3.661259  -1.318825   0.054637    0.002575  0.001003  0.002504
   C   2.930341  -0.350111   0.748856    0.002505  0.000959  0.002442
   N   3.702220   0.464559   1.439320    0.002212  0.001235  0.002431
   C   4.998330   0.044150   1.211303    0.002239  0.001237  0.002651
   C   6.209002   0.556883   1.700242    0.002071  0.001372  0.002918
   C   7.390468  -0.058295   1.293518    0.002204  0.001499  0.003105
   C   7.377622  -1.162920   0.

Step   21 : Displace = 9.843e-04/2.813e-03 (rms/max) Trust = 9.965e-04 (=) Grad = 2.212e-05/7.225e-05 (rms/max) E (change) = -2061.4583844129 (-3.487e-08) Quality = 0.083
Hessian Eigenvalues: 1.41225e-03 4.12487e-03 4.62110e-03 ... 7.11371e-01 7.66888e-01 1.01304e+00



Geometry optimization cycle 23
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   2.382746  -4.076286  -2.876203    0.000943  0.000105  0.001810
   C   2.932915  -4.533815  -1.675085    0.001009  0.000331  0.001864
   C   3.347513  -3.623016  -0.700177    0.001170  0.000463  0.001673
   C   3.213319  -2.249967  -0.932973    0.001255  0.000420  0.001390
   C   2.676199  -1.786165  -2.139032    0.001193  0.000234  0.001342
   C   2.254439  -2.702538  -3.105013    0.001038  0.000074  0.001566
   N   3.662669  -1.318250   0.055801    0.001410  0.000575  0.001164
   C   2.931666  -0.349492   0.749825    0.001326  0.000620  0.000968
   N   3.703528   0.465326   1.440136    0.001308  0.000767  0.000816
   C   4.999642   0.044817   1.212302    0.001312  0.000667  0.000999
   C   6.210317   0.557551   1.701225    0.001315  0.000668  0.000983
   C   7.391855  -0.057648   1.294729    0.001387  0.000647  0.001211
   C   7.379060  -1.162405   0.

Step   22 : Displace = 5.306e-04/1.224e-03 (rms/max) Trust = 4.921e-04 (-) Grad = 1.751e-05/4.410e-05 (rms/max) E (change) = -2061.4583843652 (+4.772e-08) Quality = -0.149
Hessian Eigenvalues: 1.41225e-03 4.12487e-03 4.62110e-03 ... 7.11371e-01 7.66888e-01 1.01304e+00
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 20176.078 seconds



WARN: 12 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -2061.45838436537
E(N@N) = -2061.458384 Hartree


In [4]:
# 3. 中性構造でのアニオン状態の単点計算 (A@N)
# 中性の構造 (mol_N_opt) を使用し、電荷-1, スピン1(Doublet) に設定
print("\n--- [2/4] 中性構造でのアニオンの単点計算 (A@N) ---")
mol_A_at_N = mol_N_opt.copy()
mol_A_at_N.charge = -1
mol_A_at_N.spin = 1
mol_A_at_N.build()

# 中性構造でのアニオンの単点エネルギー計算
# 開殻系なので UKS を使用
mf_A_at_N = uks.UKS(mol_A_at_N, xc=XC_FUNCTIONAL)
E_A_N = mf_A_at_N.kernel()
print(f"E(A@N) = {E_A_N:.6f} Hartree")

del mf_A_at_N
release_gpu()


--- [2/4] 中性構造でのアニオンの単点計算 (A@N) ---

WARN: 12 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -2061.47812062263  <S^2> = 0.75683475  2S+1 = 2.0068231
E(A@N) = -2061.478121 Hartree


In [5]:
# 4. アニオン状態の構造最適化 (A@A)
print("\n--- [3/4] アニオン状態の構造最適化 (A@A) ---")
# mf_A を初期状態として構造最適化を開始
mf_A = uks.UKS(mol_A_at_N, xc=XC_FUNCTIONAL)
mol_A_opt = optimize(mf_A, maxsteps=100)
del mf_A
release_gpu()

# アニオン状態のエネルギー計算（最適化構造でのエネルギーを取得するため、再度SP計算を実行。）
mf_A_opt = uks.UKS(mol_A_opt, xc=XC_FUNCTIONAL)
E_A_A = mf_A_opt.kernel()
print(f"E(A@A) = {E_A_A:.6f} Hartree")

# アニオン最適化構造の保存
xyz_A_opt = os.path.join(out_dir, f"{MOL_NAME}_Anion_opt.xyz")
with open(xyz_A_opt, "w") as f:
    f.write(mol_A_opt.tostring(format='xyz'))

del mf_A_opt
release_gpu()

geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3809c46b095f182629549aa658ba98d0643d4db64.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%


--- [3/4] アニオン状態の構造最適化 (A@A) ---


243 internal coordinates being used (instead of 243 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-6
Distance 1-52
Distance 2-3
Distance 2-53
Distance 3-4
Distance 3-54
Distance 4-5
Distance 4-7
Distance 5-6
Distance 5-55
Distance 6-56
Distance 7-8
Distance 7-15
Distance 8-9
Distance 8-16
Distance 9-10
Distance 10-11
Distance 10-15
Distance 11-12
Distance 11-57
Distance 12-13
Distance 12-58
Distance 13-14
Distance 13-59
Distance 14-15
Distance 14-60
Distance 16-17
Distance 16-21
Distance 17-18
Distance 17-61
Distance 18-19
Distance 18-37
Distance 19-20
Distance 19-62
Distance 20-21
Distance 20-22
Distance 21-63
Distance 22-23
Distance 22-30
Distance 23-24
Distance 24-25
Distance 24-29
Distance 25-26
Distance 25-64
Distance 26-27
Distance 26-65
Distance 27-28
Distance 27-66
Distance 28-29
Distance 28-67
Distance 29-30
Distance 30-31
Distance 31-32
Distance 31-36
Distance 32-33
Distance 32-68
Distance 33-34
Distance 33-69
Distance 34-35
Distance 3


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   2.382746  -4.076286  -2.876203    0.000000  0.000000  0.000000
   C   2.932915  -4.533815  -1.675085    0.000000  0.000000  0.000000
   C   3.347513  -3.623016  -0.700177    0.000000  0.000000  0.000000
   C   3.213319  -2.249967  -0.932973    0.000000  0.000000  0.000000
   C   2.676199  -1.786165  -2.139032    0.000000  0.000000  0.000000
   C   2.254439  -2.702538  -3.105013    0.000000  0.000000  0.000000
   N   3.662669  -1.318250   0.055801    0.000000  0.000000 -0.000000
   C   2.931666  -0.349492   0.749825    0.000000  0.000000 -0.000000
   N   3.703528   0.465326   1.440136    0.000000 -0.000000  0.000000
   C   4.999642   0.044817   1.212302    0.000000 -0.000000  0.000000
   C   6.210317   0.557551   1.701225    0.000000  0.000000  0.000000
   C   7.391855  -0.057648   1.294729    0.000000  0.000000  0.000000
   C   7.379060  -1.162405   0.4

Step    0 : Gradient = 8.587e-03/3.410e-02 (rms/max) Energy = -2061.4781206226
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 6.02124e-01 6.02158e-01 6.09527e-01


...


Geometry optimization cycle 35
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   2.345403  -3.697881  -3.189868    0.002676  0.000781  0.000717
   C   3.006840  -4.292914  -2.112906    0.002620  0.000611  0.000658
   C   3.446460  -3.513519  -1.038543    0.002829  0.000698  0.000500
   C   3.225371  -2.131513  -1.045659    0.003097  0.000747  0.000373
   C   2.568470  -1.529784  -2.125817    0.003177  0.000942  0.000442
   C   2.126122  -2.314971  -3.191596    0.002977  0.000834  0.000606
   N   3.708539  -1.334175   0.035347    0.003384  0.000700  0.000294
   C   3.004877  -0.408259   0.822304    0.003392  0.000463  0.000551
   N   3.808148   0.296328   1.595154    0.003528  0.000498  0.000363
   C   5.086661  -0.152855   1.329897    0.003556  0.000765  0.000000
   C   6.312694   0.259924   1.869786    0.003703  0.000773 -0.000356
   C   7.477415  -0.354752   1.409460    0.003637  0.000963 -0.000798
   C   7.430936  -1.362154   0.

Step   34 : Displace = 1.233e-03/2.839e-03 (rms/max) Trust = 1.342e-03 (=) Grad = 2.414e-05/8.678e-05 (rms/max) E (change) = -2061.4846989722 (-1.358e-07) Quality = 0.138
Hessian Eigenvalues: 9.35192e-04 1.88764e-03 4.54876e-03 ... 8.12989e-01 8.78659e-01 9.33127e-01



Geometry optimization cycle 36
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   C   2.346265  -3.696933  -3.189977    0.000861  0.000949 -0.000109
   C   3.007629  -4.292119  -2.113048    0.000789  0.000795 -0.000142
   C   3.447369  -3.512968  -1.038566    0.000909  0.000552 -0.000023
   C   3.226429  -2.130938  -1.045565    0.001057  0.000575  0.000094
   C   2.569606  -1.529086  -2.125707    0.001136  0.000697  0.000111
   C   2.127123  -2.314004  -3.191626    0.001001  0.000966 -0.000029
   N   3.709678  -1.333739   0.035507    0.001139  0.000436  0.000159
   C   3.006064  -0.407922   0.822644    0.001187  0.000338  0.000341
   N   3.809459   0.296539   1.595491    0.001311  0.000211  0.000337
   C   5.087957  -0.152594   1.330082    0.001296  0.000260  0.000184
   C   6.314016   0.260132   1.869960    0.001322  0.000208  0.000175
   C   7.478692  -0.354494   1.409469    0.001277  0.000258  0.000009
   C   7.431994  -1.361755   0.

Step   35 : Displace = 6.247e-04/1.545e-03 (rms/max) Trust = 6.166e-04 (-) Grad = 1.637e-05/3.951e-05 (rms/max) E (change) = -2061.4846991231 (-1.508e-07) Quality = 0.328
Hessian Eigenvalues: 9.35192e-04 1.88764e-03 4.54876e-03 ... 8.12989e-01 8.78659e-01 9.33127e-01
Converged! =D

    #==========================================================================#
    #| If this code has benefited your research, please support us by citing: |#
    #|                                                                        |#
    #| Wang, L.-P.; Song, C.C. (2016) "Geometry optimization made simple with |#
    #| translation and rotation coordinates", J. Chem, Phys. 144, 214108.     |#
    #| http://dx.doi.org/10.1063/1.4952956                                    |#
    #==========================================================================#
    Time elapsed since start of run_optimizer: 45018.415 seconds



WARN: 13 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -2061.48469912268  <S^2> = 0.76004345  2S+1 = 2.0100184
E(A@A) = -2061.484699 Hartree


In [6]:
# 5. アニオン構造での中性状態の単点計算 (N@A)
print("\n--- [4/4] アニオン構造での中性状態の単点計算 (N@A) ---")
mol_N_at_A = mol_A_opt.copy()
mol_N_at_A.charge = 0
mol_N_at_A.spin = 0
mol_N_at_A.build()

# アニオン構造での中性状態の単点エネルギー計算
mf_N_at_A = rks.RKS(mol_N_at_A, xc=XC_FUNCTIONAL)
E_N_A = mf_N_at_A.kernel()
print(f"E(N@A) = {E_N_A:.6f} Hartree")

del mf_N_at_A
release_gpu()


--- [4/4] アニオン構造での中性状態の単点計算 (N@A) ---

WARN: 13 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.

converged SCF energy = -2061.45118823668
E(N@A) = -2061.451188 Hartree


In [7]:
# 6. 結果の計算と出力
# Hartree to eV conversion factor
H2EV = 27.211386245988

vEA = (E_N_N - E_A_N) * H2EV
aEA = (E_N_N - E_A_A) * H2EV
lambda_1 = (E_A_N - E_A_A) * H2EV
lambda_2 = (E_N_A - E_N_N) * H2EV
lambda_e = lambda_1 + lambda_2

output_text = f"""
{'='*40}
【{MOL_NAME} の EA/Reorg 計算結果】
Level of theory: {XC_FUNCTIONAL} / {BASIS_SET}
E(N@N) : {E_N_N:12.6f} Hartree
E(A@N) : {E_A_N:12.6f} Hartree
E(A@A) : {E_A_A:12.6f} Hartree
E(N@A) : {E_N_A:12.6f} Hartree
{'-' * 40}
垂直EA (vEA)          : {vEA:.4f} eV
断熱EA (aEA)          : {aEA:.4f} eV
緩和エネルギー1 (λ1) : {lambda_1:.4f} eV
緩和エネルギー2 (λ2) : {lambda_2:.4f} eV
電子再配列エネルギー  : {lambda_e:.4f} eV
{'='*40}
"""
print(output_text)

# 結果を result.txt に出力
result_path = os.path.join(out_dir, "result.txt")
with open(result_path, "w", encoding="utf-8") as f_res:
    f_res.write(output_text)
print(f"結果を {result_path} に保存しました。")


【TPBi_EA の EA/Reorg 計算結果】
Level of theory: b3lyp / 6-31+g*
E(N@N) : -2061.458384 Hartree
E(A@N) : -2061.478121 Hartree
E(A@A) : -2061.484699 Hartree
E(N@A) : -2061.451188 Hartree
----------------------------------------
垂直EA (vEA)          : 0.5371 eV
断熱EA (aEA)          : 0.7161 eV
緩和エネルギー1 (λ1) : 0.1790 eV
緩和エネルギー2 (λ2) : 0.1958 eV
電子再配列エネルギー  : 0.3748 eV

結果を output/TPBi_EA/result.txt に保存しました。
